# Bmad cu_inj to Impact cu_inj

In [1]:
from pytao import TaoModel
from pprint import pprint
import json

In [2]:
M = TaoModel('/Users/chrisonian/Code/GitHub/lcls-lattice/bmad/models/cu_inj/tao.init')
#M = TaoModel('/Users/chrisonian/Code/GitHub/facet2-lattice/bmad/models/f2e_inj/tao.init')

Initialized Tao with /var/folders/wj/lfgr01993dx79p9cm_skykbw0000gn/T/tmpccg8ixlo/tao/tao.init


In [3]:
TESTELE = 'QA01'
#TESTELE = 'QA10361'

In [4]:
M.cmd_real(f'python lat_list -no_slaves  1@0>>{TESTELE}|model real:ele.s')

array([4.90893361])

In [5]:
from pytao.tao_ctypes.util import parse_tao_python_data

In [6]:
?parse_tao_python_data

Signature: parse_tao_python_data(lines, clean_key=True)
Docstring: returns dict with data
File:      ~/Code/GitHub/pytao/pytao/tao_ctypes/util.py
Type:      function


In [18]:
def get_ele_info(ele_name, tao):
    
    dat = parse_tao_python_data(tao.cmd(f'python ele:head 1@0>>{ele_name}|model'))
    dat.update(parse_tao_python_data(tao.cmd(f'python ele:gen_attribs 1@0>>{ele_name}|model')))
    
    return dat
    
get_ele_info(TESTELE, M)    

{'universe': 1,
 '1^ix_branch': 0,
 'ix_ele': 78,
 'key': 'Quadrupole',
 'name': 'QA01',
 'type': 'ETA',
 'alias': 'QUAD:IN20:361',
 'descrip': '',
 'is_on': True,
 's': 4.90893360500005,
 's_start': 4.80093360500005,
 'ref_time': 1.64153317276854e-08,
 'has#methods': True,
 'has#ab_multipoles': True,
 'has#kt_multipoles': False,
 'has#multipoles_elec': True,
 'has#ac_kick': False,
 'has#taylor': False,
 'has#spin_taylor': False,
 'has#wake': False,
 'num#cartesian_map': 0,
 'num#cylindrical_map': 0,
 'num#taylor_field': 0,
 'num#grid_field': 0,
 'has#wall3d': 0,
 'has#control': False,
 'has#twiss': True,
 'has#mat6': True,
 'has#floor': True,
 'has#photon': False,
 'has#lord_slave': True,
 'L': 0.108,
 'units#L': 'm',
 'TILT': 0.0,
 'units#TILT': 'rad',
 'K1': -8.6773946,
 'units#K1': '1/m^2',
 'FRINGE_TYPE': 'None',
 'FRINGE_AT': 'Both_Ends',
 'SPIN_FRINGE_ON': True,
 'R0_ELEC': 0.0,
 'units#R0_ELEC': 'm',
 'R0_MAG': 0.0,
 'units#R0_MAG': 'm',
 'FQ1': 0.0,
 'units#FQ1': 'm',
 'FQ2': 

# Quads

In [8]:
from impact.lattice import ele_line

In [9]:
sample_ele = {'description': 'name:QA01',
 'original': '0.204 0 0 1 4.752856 1.8524 0.108 0.016 0.0 0.0 0.0 0.0 0.0 /!name:QA01',
 'L': 0.204,
 'type': 'quadrupole',
 'zedge': 4.752856,
 'b1_gradient': 1.8524,
 'L_effective': 0.108,
 'radius': 0.016,
 'x_offset': 0.0,
 'y_offset': 0.0,
 'x_rotation': 0.0,
 'y_rotation': 0.0,
 'z_rotation': 0.0,
 's': 4.956856,
 'name': 'QA01'}

In [10]:
# This is an actual line
ele_line(sample_ele)

'0.204 0 0 1 4.752856 1.8524 0.108 0.016 0.0 0.0 0.0 0.0 0.0 /!name:QA01'

In [20]:
def impact_quad(ele_name, tao, radius=0.016):
    """
    
    |            L            |
    | pad | L_effective | pad |
    zedge
    """
    d = get_ele_info(ele_name, tao)
    assert d['key'].lower() == 'quadrupole'
    ele = {'name':ele_name}
    ele['type'] = d['key'].lower()
    ele['L_effective'] = d['L']
    ele['radius'] = radius
    
    pad =  3*radius
    
    ele['L'] = round(d['L'] + 2*pad, 9)
    ele['zedge'] = round(d['s_start'] - pad, 9)
    ele['b1_gradient'] = d['B1_GRADIENT']
    
    ele['x_offset'] = d['X_OFFSET']
    ele['y_offset'] = d['Y_OFFSET']
    ele['z_rotation'] = d['TILT_TOT']
    return ele
iele = impact_quad(TESTELE, M)    

    

In [21]:
ele_line(iele)

'0.204 0 0 1 4.752933605 1.85239919370818 0.108 0.016 0.0 0.0 0.0 0.0 0.0 /!name:QA01'

In [22]:
TESTELES = ['QA01', 'QA02'] #, 'QE01', 'QE02', 'QE03', 'QE04']
#TESTELES = ['QA10361', 'QA10371', 'QE10425', 'QE10441', 'QE10511', 'QE10525']

lines = []
for name in TESTELES:
    iele = impact_quad(name, M)  
    line = ele_line(iele)
    lines.append(line)
for l in lines:
    print(l)

0.204 0 0 1 4.752933605 1.85239919370818 0.108 0.016 0.0 0.0 0.0 0.0 0.0 /!name:QA01
0.204 0 0 1 5.081309605 -1.85239919370818 0.108 0.016 0.0 0.0 0.0 0.0 0.0 /!name:QA02


# Write beam (at markers)

In [ ]:
sample_ele ={'description': 'name:YAG03',
 'original': '0 1 42 -2 0.0 0.0  4.614539 /!name:YAG03',
 'type': 'write_beam',
 'filename': 'fort.42',
 'sample_frequency': 1,
 's': 4.614539,
 'name': 'YAG03'}

ele_line(sample_ele)

In [ ]:
def impact_write_beam(ele_name, tao, ix=101, sample_frequency=1):
    """

    """
    d = get_ele_info(ele_name, tao)

    ele = {'name':ele_name}
    ele['type'] = 'write_beam'
    ele['sample_frequency'] = sample_frequency
    ele['filename'] = f'fort.{ix}'
    ele['s'] = d['s']
    line = ele_line(ele)
    
    return line
impact_write_beam('YAG03', M)

In [ ]:
lines = []
ii = 100
for name in ['YAG01', 'YAG02', 'YAG03','WS01','OTR1', 'WS02', 'OTR2', 'WS03', 'OTR3']:
    ii += 1
    line = impact_write_beam(name, M, ix=ii)  
    lines.append(line)
for l in lines:
    print(l)